# M6 · Belady Oracle and a simple predictor

**Belady (Oracle):** on a miss, evict the expert whose *next use is furthest in the future*.
It cheats (it reads the future trace), so it is the **best hit rate any eviction policy can reach**.
We use it as a ruler: "LRU gets X% of the way to the best possible".

**Predictor + prefetch:** guess which experts will be needed *before* the router asks,
and copy them early. A copy that happens early does not make the GPU wait.
- **CrossLayer**: layer L's experts → guess layer L+1's experts (same token)
- **NextToken**: layer L's experts for token t → guess layer L's experts for token t+1

The predictor is just a **table of counts** learned on the first half of the trace
(no neural network, no training). See `src/predictor.py`.

**Kaggle:** Accelerator = **None** (CPU is enough, saves GPU quota), Internet = On.
**Inputs:** Add Input → Datasets → **moe-traces** (from M2).
Optional: add the M3 output for the measured copy time.

In [ ]:
# ---- Setup: get the latest project code (run this cell first, every session) ----
import os, sys

REPO_URL = "https://github.com/YOUR-TEAM/moe-offload.git"   # <-- change to your team's repo
# Private repo? Add a Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

if os.path.exists("/kaggle"):                      # running on Kaggle
    import glob, shutil
    CODE_DIR = "/kaggle/working/moe-offload"
    # Option A: the code was uploaded as a Kaggle dataset ("moe-offload-code") and added as an input
    uploaded = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/moe-offload-code/config.py", recursive=True)]
    if uploaded:
        shutil.copytree(uploaded[0], CODE_DIR, dirs_exist_ok=True)
        print("using code from dataset:", uploaded[0])
    # Option B: get the code from GitHub
    elif not os.path.exists(CODE_DIR):
        !git clone -q {REPO_URL} {CODE_DIR}         # first time: download the code
    else:
        !git -C {CODE_DIR} pull -q                  # later: get the newest commits
    os.chdir(CODE_DIR)
    !pip install -q -r requirements.txt
else:                                              # running on your own computer
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                             # go to the repo root

sys.path.insert(0, os.getcwd())                    # so "import config" and "from src..." work
!git log -1 --format="Code version: %h  %s"

## 1. Belady vs LRU on a tiny example
Pattern 1, 2, 3, 1, 2, 3 with 2 slots: LRU always throws out exactly the expert needed next.

In [ ]:
import numpy as np
from src.simulator import simulate

tiny = np.array([1, 2, 3, 1, 2, 3]).reshape(-1, 1, 1)     # [tokens, 1 layer, 1 expert]
for policy in ["LRU", "Belady"]:
    print(policy, "hits:", simulate(tiny, policy, slots=2, num_experts=8)["hits"])

## 2. How good are the predictors?
**Precision** = fraction of the guessed experts that were really needed.

In [ ]:
import config
from src.files import load_trace
from src.simulator import split_trace
from src.predictor import make_predictor

fit, test = split_trace(load_trace("wikitext"))
for name in ["CrossLayer", "NextToken"]:
    pred = make_predictor(name).fit(fit)                  # count on the FIT half only
    r = simulate(test[:10_000], "LRU", 16, predictor=pred, prefetch_p=config.PREFETCH_P)
    print(f"{name:10s} precision of {config.PREFETCH_P} guesses: {r['prediction_precision']:.2f}  "
          f"(random guess would be 8/64 = 0.125)")

## 3. The full comparison
All policies × cache sizes × datasets. Random is repeated with 5 seeds and averaged.
This takes ~1 hour on the Kaggle CPU. Use `max_test_tokens=20_000` for a quick run.

Columns:
- `misses`: copies the GPU must **wait** for
- `prefetches`: copies done **early** (no waiting, but they still use bandwidth/energy)
- `stall_ms_per_token`: estimated waiting time per token = misses × copy time / tokens

In [ ]:
import os
import pandas as pd
from src.experiments import run_grid

MAX_TEST_TOKENS = None         # e.g. 20_000 for a quick run
results = pd.concat([run_grid(load_trace(d), d, max_test_tokens=MAX_TEST_TOKENS)
                     for d in ["wikitext", "code"]])
results.to_csv(os.path.join(config.RESULTS_DIR, "policy_results.csv"), index=False)
results.pivot_table(index=["dataset", "slots"], columns="variant", values="hit_rate", sort=False).round(3)

## 4. How close does each policy get to the Oracle?
0% = as good as Random, 100% = as good as Belady.
- With 8 slots all eviction policies are equal, so this shows NaN (0 / 0).
- Prefetching can score **above 100%**: Belady is the best policy that only *evicts*,
  while prefetching also copies experts in early.

In [ ]:
def gap_closed(group):
    """0% = as bad as Random, 100% = as good as Belady."""
    rnd = group.loc[group.variant == "Random", "hit_rate"].item()
    best = group.loc[group.variant == "Belady", "hit_rate"].item()
    return pd.Series(100 * (group.hit_rate.values - rnd) / (best - rnd), index=group.variant)

closed = results.groupby(["dataset", "slots"]).apply(gap_closed).round(0)
closed[["LRU", "LFU", "LRU+CrossLayer", "LRU+NextToken"]]

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, dataset in zip(axes, ["wikitext", "code"]):
    sub = results[results.dataset == dataset]
    for variant, g in sub.groupby("variant", sort=False):
        ax.plot(100 * g.slots / 64, g.stall_ms_per_token, marker="o", label=variant)
    ax.set(title=f"{dataset}: waiting time for copies", xlabel="cache size (% of experts)",
           ylabel="ms per token", yscale="log")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(config.RESULTS_DIR, "M6_stall.png"), dpi=130); plt.show()

## ✅ Done when
`results/policy_results.csv` exists with all 7 variants. Write in `docs/M6.md`:
which policy wins, how close it is to Belady, and whether prefetching reduces waiting time.